In [ ]:
import os
import sys

import cv2
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))  # repo root, so `src` imports work


In [ ]:
from src.cryptosystem import Cryptosystem, Encryption, Decryption

In [ ]:
image=cv2.imread('../outputs/fruits1024/fruits1024_cipher.png')
image_name = "fruits1024"
x = Cryptosystem(image)

In [ ]:
k = "afcd414b5b529546ced95f961150c58dbf99faff49f3ab820ec9841e88ad"
kd = "bfcd414b5b529546ced95f961150c58dbf99faff49f3ab820ec9841e88ad"

In [ ]:
def hexadecimal_to_binary(hex_string):
    decimal_number = int(hex_string, 16)  # Convert hexadecimal string to decimal integer
    binary_string = bin(decimal_number)[2:]  # Convert decimal integer to binary string
    return binary_string



In [ ]:
print(hexadecimal_to_binary(k))
print("\n ======== \n")
print(hexadecimal_to_binary(kd))

In [ ]:
new_key = hexadecimal_to_binary(kd)

In [ ]:
a1,b1,x0,y0 = x.generateInitials(new_key)
print(a1,b1,x0,y0)

In [ ]:
map1 = x.getChaoticMap(a1,b1,x0,y0)
map2 = x.getChaoticMap(a1,b1,map1[len(map1)-1][0],map1[len(map1)-1][1]) # map2 x0,y0 from the last values of map1 as given in paper
map3 = x.getChaoticMap(a1,b1,map2[len(map2)-1][0],map2[len(map2)-1][1])
map4 = x.getChaoticMap(a1,b1,map3[len(map3)-1][0],map3[len(map3)-1][1])
map5 = x.getChaoticMap(a1,b1,map4[len(map4)-1][0],map4[len(map4)-1][1])
map6 = x.getChaoticMap(a1,b1,map5[len(map5)-1][0],map5[len(map5)-1][1])

In [ ]:
IV = x.getfloorIV(8,map1)

In [ ]:
rm1 = x.getRuleMap(map2,8)
rm2 = x.getRuleMap(map3,8)
rm3 = x.getRuleMap(map5,8)
rm4 = x.getRuleMap(map6,8)

In [ ]:
DSM = x.getDSMMap(map4,rm2)

## Decryption

In [ ]:
d=Decryption(image)

In [ ]:
r,g,b=d.matrices()

In [ ]:
ered,eblue,egreen   = d.encoding(r,g,b,rm4)

In [ ]:
sred,sblue,sgreen = [],[],[]
sred   = d.rSubstitution(ered,DSM,rm3)
sblue  = d.rSubstitution(eblue,DSM,rm3)
sgreen = d.rSubstitution(egreen,DSM,rm3)

In [ ]:
dred   = d.Decoding(sred,rm1)
dblue  = d.Decoding(sblue,rm1)
dgreen = d.Decoding(sgreen,rm1)

In [ ]:
fred = d.toMatrix(dred)
fblue = d.toMatrix(dblue)
fgreen = d.toMatrix(dgreen)

In [ ]:
dmixRed = d.rSpiralMixing(fred)
dmixBlue = d.rSpiralMixing(fblue)
dmixGreen = d.rSpiralMixing(fgreen)

In [ ]:
dacmr,dacmg,dacmb = d.applyRevACM_numba(dmixRed,dmixGreen,dmixBlue)

In [ ]:
dfinalR = d.rmixRow(dacmr,IV)
dfinalG = d.rmixRow(dacmg,IV)
dfinalB = d.rmixRow(dacmb,IV)

In [ ]:
decodedImage=d.getDecodedImage(dfinalR,dfinalG,dfinalB)

In [ ]:
import os
cipherd = cv2.merge((dfinalB,dfinalG,dfinalR))
ks_directory = f"../outputs/key_sensitivity"
os.makedirs(ks_directory, exist_ok=True)  # Ensure directory exists

file_path = os.path.join(ks_directory, f"{image_name}_ks.png")
success = cv2.imwrite(file_path, cipherd)
if success:
    print(f"Image saved successfully at {file_path}")
else:
    print("Failed to save image. Check if the image is valid.")